# DuckPD GPU Vector Search & Streaming Text Embeddings Walkthrough

This interactive notebook demonstrates explicit PyTorch GPU model preparation, bounded in-engine corpus embedding, persisted embedding metadata, and exact text search over remote and local Parquet datasets.

### Highlights
- **PyTorch GPU inference**: Run the pinned BGE model through `TransformersEmbeddingProvider` on NVIDIA CUDA or AMD ROCm.
- **Explicit accelerator selection**: Request `device="cuda"`; DuckPD raises instead of silently falling back to CPU.
- **Lazy remote streaming**: Scan Parquet directly over HTTPS without downloading the full source first.
- **Persisted model identity**: DuckPD writes a metadata sidecar with the Parquet vectors and restores it on read.
- **Sidecar-Inferred Search**: Omit `model=` after reload while retaining an optional explicit fingerprint assertion.


## 1. Select the ROCm kernel and configure the model

In VS Code, choose **Select Kernel** in the upper-right and select **DuckPD ROCm 7.2.4**. If it is absent, follow the environment and kernel-registration commands in `demo/generate_data/README.md`, then reload the VS Code window. The normal repository `.venv` intentionally does not include accelerator-specific packages.

The first code cell verifies the selected interpreter, imports both GPU dependencies, and confirms that PyTorch can see the GPU before model preparation. On AMD ROCm, PyTorch exposes the device through its `cuda` API.

The Transformers backend uses a distinct model fingerprint and output file, preventing its vectors from being mixed with prior FastEmbed CPU output.

In [1]:
import importlib
import sys
from pathlib import Path
from time import perf_counter

import duckpd as pd

try:
    torch = importlib.import_module("torch")
    importlib.import_module("transformers")
except (ImportError, OSError) as error:
    raise RuntimeError(
        "This notebook is running the wrong VS Code kernel. "
        f"Current interpreter: {sys.executable}. Select Kernel > DuckPD ROCm 7.2.4, "
        "then restart the kernel. Setup instructions: demo/generate_data/README.md"
    ) from error
if not torch.cuda.is_available():
    raise RuntimeError(
        f"PyTorch in {sys.executable} cannot access a GPU. Select the DuckPD ROCm 7.2.4 "
        "kernel and restart it; DuckPD will not fall back to CPU."
    )

DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"
DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path(".")
EMBEDDED_DATA = DEMO_DIR / "nvidia-news-embedded-transformers.parquet"
QUERY = "AI chip demand and revenue growth"
TRANSFORMER_BATCH_SIZE = 64

MODEL = pd.embedding_model(
    "BAAI/bge-small-en-v1.5",
    revision="5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",
    dimension=384,
    backend="transformers",
    pooling="cls",
)

print(f"Kernel interpreter: {sys.executable}")
print(f"GPU: {torch.cuda.get_device_name(0)} (ROCm {torch.version.hip})")
print(f"DuckPD version: {pd.__version__}")
print(
    f"Embedding model: {MODEL.model} "
    f"(revision: {MODEL.revision[:12]}..., dimension: {MODEL.dimension})"
)
print(f"Embedded dataset: {EMBEDDED_DATA}")

Kernel interpreter: /home/hi/duckpd/demo/generate_data/.venv-rocm/bin/python
GPU: AMD Radeon Graphics (ROCm 7.2.53211-97f5574fe2)
DuckPD version: 0.1.4
Embedding model: BAAI/bge-small-en-v1.5 (revision: 5c38ec7c405e..., dimension: 384)
Embedded dataset: nvidia-news-embedded-transformers.parquet


## 2. Prepare the model on the GPU

Register an explicit GPU provider before preparation. `device="cuda"` selects either NVIDIA CUDA or AMD ROCm according to the installed PyTorch build. Preparation verifies the immutable local model cache and reports the actual runtime.

In [2]:
session = pd.connect()
provider = pd.TransformersEmbeddingProvider(
    MODEL,
    device="cuda",
    batch_size=TRANSFORMER_BATCH_SIZE,
)
session.register_embedding_provider(MODEL, provider)

preparation_started = perf_counter()
prepared = session.prepare_embedding_model(MODEL)
preparation_seconds = perf_counter() - preparation_started

print(f"Backend: {prepared.backend} via {prepared.execution_providers}")
print(f"Model preparation time: {preparation_seconds:.3f}s")
print(f"Persisted model fingerprint: {MODEL.fingerprint}")

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Backend: transformers via ('PyTorchROCm',)
Model preparation time: 8.120s
Persisted model fingerprint: 0868586fea64c6a028df5fbf655fb945b5c553bae405c7f52e9678eba9bb13d7


## 3. Load or build the GPU-embedded dataset

If the Transformers-specific Parquet file exists locally, load it directly. Otherwise, DuckPD lazily scans the remote Hugging Face dataset, filters NVIDIA (`NVDA`) articles, computes embeddings on the registered GPU in bounded Arrow and model batches, and persists the result. The separate filename prevents accidental reuse of FastEmbed CPU vectors.

In [3]:
if EMBEDDED_DATA.exists():
    embedded = session.read_parquet(EMBEDDED_DATA)
    dataset_status = f"Loaded {EMBEDDED_DATA}"
else:
    build_started = perf_counter()
    news = session.read_parquet(DATA_URL)
    nvidia_news = news[news["symbol"] == "NVDA"]
    embedded = nvidia_news.embed_text(
        columns=["title", "description"],
        into="embedding",
        model=MODEL,
        batch_size=64,
        null_policy="empty",
    )
    embedded.write_parquet(EMBEDDED_DATA)
    build_seconds = perf_counter() - build_started
    dataset_status = (
        f"Created {EMBEDDED_DATA} from the remote archive in {build_seconds:.3f} seconds"
    )
    embedded = session.read_parquet(EMBEDDED_DATA)

print(f"Embedding dataset: {dataset_status}")
print(f"Columns: {embedded.columns}")

Embedding dataset: Loaded nvidia-news-embedded-transformers.parquet
Columns: ('title', 'image', 'ago', 'primarysymbol', 'primarytopic', 'publisher', 'url', 'id', 'imagedomain', 'description', 'primarytopic_url', 'publisher_logo', 'publish_date', 'on_symbol_json', 'symbol', 'source', 'embedding')


## 4. Preview the source data

Inspect a few identifying columns with bounded materialization via `head()`. The large `embedding` vectors are intentionally omitted here; they remain available in the lazy frame for search.

In [4]:
preview = embedded[["symbol", "title", "publisher", "publish_date"]].head(5)
preview

,symbol,title,publisher,publish_date
0,NVDA,"If You Only Own Nvidia for AI Exposure, You're...",The Motley Fool,"Sep 7, 2026"
1,NVDA,Nvidia Stock: Is It Still a Good Buy at $230?,The Motley Fool,"Sep 7, 2026"
2,NVDA,Berkshire Hathaway Owns AI Exposure in 3 Diffe...,The Motley Fool,"Sep 7, 2026"
3,NVDA,"Broadcom Drops 10% in 3 Months: Buy, Sell or H...",Zacks,"Sep 7, 2026"
4,NVDA,Buy These 5 Semiconductor Stocks as Sales Skyr...,Zacks,"Sep 7, 2026"


## 5. Run Sidecar-Inferred Vector Search

`write_parquet()` persisted the model identity beside the vector data, and `read_parquet()` restored it. The first search therefore omits `model=`. The second passes the model explicitly as a fingerprint compatibility assertion; it does not override the column metadata. Both plans must return identical results.


In [5]:
query_started = perf_counter()
matches = embedded.vector.search_text(
    QUERY,
    column="embedding",
    metric="cosine",
    k=5,
    tie_breaker="title",
)[["symbol", "title", "publisher", "publish_date", "_distance"]]

explicit_matches = embedded.vector.search_text(
    QUERY,
    column="embedding",
    model=MODEL,
    metric="cosine",
    k=5,
    tie_breaker="title",
)[["symbol", "title", "publisher", "publish_date", "_distance"]]

logical_plan = matches.explain("json")
assert '"model_origin": "sidecar"' in logical_plan
assert QUERY not in logical_plan

result = matches.collect()
explicit_result = explicit_matches.collect()
assert result.equals(explicit_result)
query_seconds = perf_counter() - query_started

print(f"Query: {QUERY!r}")
print(f"Query-to-response for inferred and explicit plans: {query_seconds:.3f} seconds")
print("Sidecar-inferred and explicit-model results are identical.")

Query: 'AI chip demand and revenue growth'
Query-to-response for inferred and explicit plans: 0.533 seconds
Sidecar-inferred and explicit-model results are identical.


/home/hi/duckpd/demo/generate_data/.venv-rocm/lib/python3.12/site-packages/transformers/models/bert/modeling_bert.py:413: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:309.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/hi/duckpd/demo/generate_data/.venv-rocm/lib/python3.12/site-packages/transformers/models/bert/modeling_bert.py:413: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:360.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


## 6. Inspect the Results

Lower cosine distance means greater semantic similarity. The successful parity assertion above proves that sidecar inference selected the same embedding space as the explicit model. Automatic preparation is catalog-scoped; this ordinary sidecar workflow prepared the model explicitly in Step 2.


In [6]:
display(result)
session.close()

,symbol,title,publisher,publish_date,_distance
0,NVDA,Buy These 5 Semiconductor Stocks as Sales Skyr...,Zacks,"Sep 7, 2026",0.233035
1,NVDA,Better AI Infrastructure Stock: Nvidia vs. AMD,The Motley Fool,"Sep 3, 2026",0.234854
2,NVDA,Prediction: Nvidia Stock Will Double in Under ...,The Motley Fool,"Sep 6, 2026",0.235065
3,NVDA,You Could Buy Nvidia for Its 106% Revenue Grow...,The Motley Fool,"Aug 31, 2026",0.236522
4,NVDA,Upbeat AI Data Centers Demand Propels SK Hynix...,Zacks,"Sep 3, 2026",0.238219
